Setup

In [1]:
from pathlib import Path 

import numpy as np 
import pandas as pd 

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

INPUT_DIR = Path("../data/synthetic_output")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok = True)

print("Input directory:", INPUT_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Input directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\synthetic_output
Output directory: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\lead-generation\data\processed


Load raw data

In [ ]:
fact_sales_raw = pd.read_csv(INPUT_DIR/"df_fact_sales.csv")
dim_customer_raw = pd.read_csv(INPUT_DIR/"df_dim_customer.csv")
dim_product_raw = pd.read_csv(INPUT_DIR/"df_dim_product.csv")
fact_sf_raw = pd.read_csv(INPUT_DIR/"df_fact_sf.csv")

raw_shapes = pd.DataFrame({
    "table": ["fact_sales_raw", "dim_customer_raw", "dim_product_raw", "fact_sf_raw"],
    "rows": [len(fact_sales_raw), len(dim_customer_raw), len(dim_product_raw), len(fact_sf_raw)],
    "columns": [fact_sales_raw.shape[1], dim_customer_raw.shape[1], dim_product_raw.shape[1], fact_sf_raw.shape[1]]})

display(raw_shapes)

,table,rows,columns
0,fact_sales_raw,68723,5
1,dim_customer_raw,2000,10
2,dim_product_raw,180,8
3,fact_sf_raw,39508,8


In [16]:
display(fact_sales_raw.head())
display(dim_customer_raw.head())
display(dim_product_raw.head())
display(fact_sf_raw.head())

,Month_Year,Customer_ID,Product_ID,Sales,Units
0,2023-02-01,C00001,P0116,"2,301.44",13
1,2023-02-01,C00001,P0083,"1,545.15",6
2,2023-04-01,C00001,P0160,763.50,15
3,2023-05-01,C00001,P0170,805.03,19
4,2023-05-01,C00001,P0160,155.10,3


,ID_Customer,Customer_Name,Customer_Segment,Customer_Size,Industry,Region_CC,Region_OC,Country,Acquisition_Channel,Customer_Start_Date
0,C00001,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01
1,C00002,Silverline Supplies BV 0002,SMB,Small,Food & Beverage,Southern Europe,West,Germany,Partner,2022-05-01
2,C00003,Northstar Technologies Ltd 0003,SMB,Large,Construction,Western Europe,North,Netherlands,Website,2022-02-01
3,C00004,Greenfield Trading Group 0004,Mid-Market,Large,Manufacturing,Benelux,South,Belgium,Partner,2021-10-01
4,C00005,Greenfield Retail Group BV 0005,SMB,Medium,Retail,Western Europe,North,Netherlands,Website,2020-12-01


,ID_Product,Product_Name,Prod_Line,Prd_Grp_1,Prd_Grp_2,Prd_Grp_3,Unit_Price,Margin_Category
0,P0001,Controllers Type 1 0001,Safety,Automation Modules,Controllers,Controllers Type 1,583.74,Low
1,P0002,Extended Support Type 2 0002,Specialty Materials,Service Contracts,Extended Support,Extended Support Type 2,581.89,Medium
2,P0003,Repair Kits Type 1 0003,Consumables,Maintenance Kits,Repair Kits,Repair Kits Type 1,122.81,High
3,P0004,Calibration Kits Type 1 0004,Consumables,Maintenance Kits,Calibration Kits,Calibration Kits Type 1,136.53,Low
4,P0005,Sensors Type 2 0005,Consumables,Automation Modules,Sensors,Sensors Type 2,330.94,High


,Month_Year,Customer_ID,SF_Activity_Count,SF_Selling_Time,SF_Activity_Time_,Activity_Type,Sales_Rep,Opportunity_Stage
0,2023-01-01,C00001,1,0.51,0.38,Account Review,Rep_13,Prospecting
1,2023-02-01,C00001,1,0.74,0.36,Call,Rep_02,Proposal
2,2023-03-01,C00001,2,2.28,0.58,Account Review,Rep_14,Closed Lost
3,2023-05-01,C00001,12,8.09,7.56,Call,Rep_21,Proposal
4,2023-06-01,C00001,4,4.11,1.58,Demo,Rep_01,Qualification


Initital Data Quality Check

In [ ]:
def profile_table(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "table": table_name,
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_values": df.isna().sum().values,
        "missing_pct": df.isna().mean().round(4).values,
        "nunique": df.nunique(dropna=True).values})


raw_profile = pd.concat(
    [profile_table(fact_sales_raw, "fact_sales_raw"),
        profile_table(dim_customer_raw, "dim_customer_raw"),
        profile_table(dim_product_raw, "dim_product_raw"),
        profile_table(fact_sf_raw, "fact_sf_raw")],
    ignore_index=True)

display(raw_profile)


,table,column,dtype,missing_values,missing_pct,nunique
0,fact_sales_raw,Month_Year,object,0,0.00,36
1,fact_sales_raw,Customer_ID,object,0,0.00,1993
2,fact_sales_raw,Product_ID,object,0,0.00,180
3,fact_sales_raw,Sales,float64,343,0.01,59116
4,fact_sales_raw,Units,int64,0,0.00,61
5,dim_customer_raw,ID_Customer,object,0,0.00,2000
6,dim_customer_raw,Customer_Name,object,0,0.00,2000
7,dim_customer_raw,Customer_Segment,object,0,0.00,5
8,dim_customer_raw,Customer_Size,object,0,0.00,4
9,dim_customer_raw,Industry,object,0,0.00,10


In [ ]:
duplicate_summary = pd.DataFrame({
    "table": ["fact_sales_raw", "dim_customer_raw", "dim_product_raw", "fact_sf_raw"],
    "duplicate_rows": [
        fact_sales_raw.duplicated().sum(),
        dim_customer_raw.duplicated().sum(),
        dim_product_raw.duplicated().sum(),
        fact_sf_raw.duplicated().sum()]})

display(duplicate_summary)

,table,duplicate_rows
0,fact_sales_raw,0
1,dim_customer_raw,0
2,dim_product_raw,0
3,fact_sf_raw,0


Helper Functions

In [6]:
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    out=df.copy()
    out.columns = [str(c) for c in out.columns]
    return out 

def assert_required_columns(df: pd.DataFrame, required: list[str], table_name: str) -> None: 
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")
    
def show_cleaning_result(raw: pd.DataFrame, clean: pd.DataFrame, table_name: str) -> None:
    summary = pd.DataFrame({
        "metric": ["rows", "columns", "duplicate_rows", "missing_cells"],
        "raw": [len(raw), raw.shape[1], raw.duplicated().sum(), raw.isna().sum().sum()],
        "clean": [len(clean), clean.shape[1], clean.duplicated().sum(), clean.isna().sum().sum()]
    })

    print(table_name)
    display(summary)
    display(clean.head())


Clean Sales Fact Table

In [10]:
def clean_fact_sales(fact_sales: pd.DataFrame) -> pd.DataFrame:
    required = ["Month_Year", "Customer_ID", "Product_ID", "Sales", "Units"]
    assert_required_columns(fact_sales, required, "fact_sales")

    df = normalize_column_names(fact_sales)
    df["Month_Year"] = pd.to_datetime(df["Month_Year"], errors="coerce")
    df["Customer_ID"] = df["Customer_ID"].astype("string").str.strip()
    df["Product_ID"] = df["Product_ID"].astype("string").str.strip()
    df["Sales"]=pd.to_numeric(df["Sales"], errors="coerce")
    df["Units"]=pd.to_numeric(df["Units"], errors="coerce")

    df = df.dropna(subset=["Month_Year", "Customer_ID", "Product_ID"])
    df = df[df["Units"].fillna(0) >= 0]
    df = df[df["Sales"].fillna(0) >= 0]

    unit_price_proxy = df["Sales"] / df["Units"].replace(0, np.nan)
    median_unit_price = unit_price_proxy.median()
    df["Sales"] = df["Sales"].fillna((df["Units"] * median_unit_price).round(2))
    df["Units"] = df["Units"].fillna(0).astype(int)

    df["YearMonth"] = df["Month_Year"].dt.to_period("M")
    df["Year"] = df["Month_Year"].dt.year
    df["Month"] = df["Month_Year"].dt.month 

    df = df.rename(columns={"Month_Year": "Date", "Customer_ID": "ID_Customer", "Product_ID": "ID_Product"})

    return df.drop_duplicates()

fact_sales_clean = clean_fact_sales(fact_sales_raw)
show_cleaning_result(fact_sales_raw, fact_sales_clean, "fact_sales_clean")


fact_sales_clean


,metric,raw,clean
0,rows,68723,68723
1,columns,5,8
2,duplicate_rows,0,0
3,missing_cells,343,0


,Date,ID_Customer,ID_Product,Sales,Units,YearMonth,Year,Month
0,2023-02-01,C00001,P0116,"2,301.44",13,2023-02,2023,2
1,2023-02-01,C00001,P0083,"1,545.15",6,2023-02,2023,2
2,2023-04-01,C00001,P0160,763.50,15,2023-04,2023,4
3,2023-05-01,C00001,P0170,805.03,19,2023-05,2023,5
4,2023-05-01,C00001,P0160,155.10,3,2023-05,2023,5


Clean Customer Dimension

In [ ]:
def clean_dim_customer(dim_customer: pd.DataFrame) -> pd.DataFrame:
    required = [
        "ID_Customer", "Customer_Name", "Customer_Segment", "Customer_Size",
        "Industry", "Region_CC", "Region_OC", "Country",
        "Acquisition_Channel", "Customer_Start_Date",
    ]
    assert_required_columns(dim_customer, required, "dim_customer")

    df = normalize_column_names(dim_customer)
    df["ID_Customer"] = df["ID_Customer"].astype("string").str.strip()
    df["Customer_Start_Date"] = pd.to_datetime(df["Customer_Start_Date"], errors="coerce")

    text_cols = [
        "Customer_Name", "Customer_Segment", "Customer_Size", "Industry",
        "Region_CC", "Region_OC", "Country", "Acquisition_Channel"]
    for col in text_cols:
        df[col] = df[col].astype("string").str.strip()

    fill_values = {
        "Region_OC": "Unknown",
        "Region_CC": "Unknown",
        "Acquisition_Channel": "Unknown",
        "Customer_Segment": "Unknown",
        "Customer_Size": "Unknown",
        "Industry": "Unknown",
        "Country": "Unknown"}
    df = df.fillna(fill_values)
    df = df.sort_values("ID_Customer").drop_duplicates(subset=["ID_Customer"], keep="first")
    return df


dim_customer_clean = clean_dim_customer(dim_customer_raw)
show_cleaning_result(dim_customer_raw, dim_customer_clean, "dim_customer_clean")


dim_customer_clean


,metric,raw,clean
0,rows,2000,2000
1,columns,10,10
2,duplicate_rows,0,0
3,missing_cells,100,0


,ID_Customer,Customer_Name,Customer_Segment,Customer_Size,Industry,Region_CC,Region_OC,Country,Acquisition_Channel,Customer_Start_Date
0,C00001,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01
1,C00002,Silverline Supplies BV 0002,SMB,Small,Food & Beverage,Southern Europe,West,Germany,Partner,2022-05-01
2,C00003,Northstar Technologies Ltd 0003,SMB,Large,Construction,Western Europe,North,Netherlands,Website,2022-02-01
3,C00004,Greenfield Trading Group 0004,Mid-Market,Large,Manufacturing,Benelux,South,Belgium,Partner,2021-10-01
4,C00005,Greenfield Retail Group BV 0005,SMB,Medium,Retail,Western Europe,North,Netherlands,Website,2020-12-01


Clean Product Dimension

In [ ]:
def clean_dim_product(dim_product: pd.DataFrame) -> pd.DataFrame:
    required = [
        "ID_Product", "Product_Name", "Prod_Line", "Prd_Grp_1",
        "Prd_Grp_2", "Prd_Grp_3", "Unit_Price", "Margin_Category"]
    assert_required_columns(dim_product, required, "dim_product")

    df = normalize_column_names(dim_product)
    df["ID_Product"] = df["ID_Product"].astype("string").str.strip()
    df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce")

    text_cols = ["Product_Name", "Prod_Line", "Prd_Grp_1", "Prd_Grp_2", "Prd_Grp_3", "Margin_Category"]
    for col in text_cols:
        df[col] = df[col].astype("string").str.strip()

    df["Unit_Price"] = df["Unit_Price"].fillna(df["Unit_Price"].median())
    df["Margin_Category"] = df["Margin_Category"].fillna("Unknown")
    df = df.sort_values("ID_Product").drop_duplicates(subset=["ID_Product"], keep="first")
    return df


dim_product_clean = clean_dim_product(dim_product_raw)
show_cleaning_result(dim_product_raw, dim_product_clean, "dim_product_clean")


dim_product_clean


,metric,raw,clean
0,rows,180,180
1,columns,8,8
2,duplicate_rows,0,0
3,missing_cells,2,0


,ID_Product,Product_Name,Prod_Line,Prd_Grp_1,Prd_Grp_2,Prd_Grp_3,Unit_Price,Margin_Category
0,P0001,Controllers Type 1 0001,Safety,Automation Modules,Controllers,Controllers Type 1,583.74,Low
1,P0002,Extended Support Type 2 0002,Specialty Materials,Service Contracts,Extended Support,Extended Support Type 2,581.89,Medium
2,P0003,Repair Kits Type 1 0003,Consumables,Maintenance Kits,Repair Kits,Repair Kits Type 1,122.81,High
3,P0004,Calibration Kits Type 1 0004,Consumables,Maintenance Kits,Calibration Kits,Calibration Kits Type 1,136.53,Low
4,P0005,Sensors Type 2 0005,Consumables,Automation Modules,Sensors,Sensors Type 2,330.94,High


Clean Salesforce Activity Fact Table

In [ ]:
def clean_fact_sf(fact_sf: pd.DataFrame) -> pd.DataFrame:
    required = [
        "Month_Year", "Customer_ID", "SF_Activity_Count",
        "SF_Selling_Time", "SF_Activity_Time_", "Activity_Type",
        "Sales_Rep", "Opportunity_Stage"]
    assert_required_columns(fact_sf, required, "fact_sf")

    df = normalize_column_names(fact_sf)
    df["Month_Year"] = pd.to_datetime(df["Month_Year"], errors="coerce")
    df["Customer_ID"] = df["Customer_ID"].astype("string").str.strip()

    numeric_cols = ["SF_Activity_Count", "SF_Selling_Time", "SF_Activity_Time_"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    text_cols = ["Activity_Type", "Sales_Rep", "Opportunity_Stage"]
    for col in text_cols:
        df[col] = df[col].astype("string").str.strip().fillna("Unknown")

    df = df.dropna(subset=["Month_Year", "Customer_ID"])
    df["YearMonth"] = df["Month_Year"].dt.to_period("M")
    df["Year"] = df["Month_Year"].dt.year
    df["Month"] = df["Month_Year"].dt.month

    df = df.rename(columns={"Month_Year": "Date", "Customer_ID": "ID_Customer"})
    return df.drop_duplicates()


fact_sf_clean = clean_fact_sf(fact_sf_raw)
show_cleaning_result(fact_sf_raw, fact_sf_clean, "fact_sf_clean")


fact_sf_clean


,metric,raw,clean
0,rows,39508,39508
1,columns,8,11
2,duplicate_rows,0,0
3,missing_cells,790,0


,Date,ID_Customer,SF_Activity_Count,SF_Selling_Time,SF_Activity_Time_,Activity_Type,Sales_Rep,Opportunity_Stage,YearMonth,Year,Month
0,2023-01-01,C00001,1,0.51,0.38,Account Review,Rep_13,Prospecting,2023-01,2023,1
1,2023-02-01,C00001,1,0.74,0.36,Call,Rep_02,Proposal,2023-02,2023,2
2,2023-03-01,C00001,2,2.28,0.58,Account Review,Rep_14,Closed Lost,2023-03,2023,3
3,2023-05-01,C00001,12,8.09,7.56,Call,Rep_21,Proposal,2023-05,2023,5
4,2023-06-01,C00001,4,4.11,1.58,Demo,Rep_01,Qualification,2023-06,2023,6


Referential Integrity Checks

In [ ]:
checks = {
    "sales_unknown_customers": len(set(fact_sales_clean["ID_Customer"]) - set(dim_customer_clean["ID_Customer"])),
    "sales_unknown_products": len(set(fact_sales_clean["ID_Product"]) - set(dim_product_clean["ID_Product"])),
    "sf_unknown_customers": len(set(fact_sf_clean["ID_Customer"]) - set(dim_customer_clean["ID_Customer"])),
    "duplicate_customers": int(dim_customer_clean["ID_Customer"].duplicated().sum()),
    "duplicate_products": int(dim_product_clean["ID_Product"].duplicated().sum()),
    "negative_sales_rows": int((fact_sales_clean["Sales"] < 0).sum()),
    "negative_units_rows": int((fact_sales_clean["Units"] < 0).sum())}

validation_report = pd.DataFrame(checks.items(), columns=["check", "value"])
display(validation_report)

if validation_report["value"].sum() != 0:
    raise ValueError("Referential integrity or numeric validation failed.")


,check,value
0,sales_unknown_customers,0
1,sales_unknown_products,0
2,sf_unknown_customers,0
3,duplicate_customers,0
4,duplicate_products,0
5,negative_sales_rows,0
6,negative_units_rows,0


Create Full Sales Table 

In [19]:
df_sales_full = (
    fact_sales_clean
    .merge(dim_customer_clean, on="ID_Customer", how="left", validate="many_to_one")
    .merge(dim_product_clean, on="ID_Product", how="left", validate="many_to_one"))

print("df_sales_full shape:", df_sales_full.shape)
display(df_sales_full.head())


df_sales_full shape: (68723, 24)


,Date,ID_Customer,ID_Product,Sales,Units,YearMonth,Year,Month,Customer_Name,Customer_Segment,Customer_Size,Industry,Region_CC,Region_OC,Country,Acquisition_Channel,Customer_Start_Date,Product_Name,Prod_Line,Prd_Grp_1,Prd_Grp_2,Prd_Grp_3,Unit_Price,Margin_Category
0,2023-02-01,C00001,P0116,"2,301.44",13,2023-02,2023,2,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Premium Standard Type 2 0116,Consumables,Premium Products,Premium Standard,Premium Standard Type 2,194.35,Low
1,2023-02-01,C00001,P0083,"1,545.15",6,2023-02,2023,2,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Premium Plus Type 1 0083,Maintenance,Premium Products,Premium Plus,Premium Plus Type 1,285.90,High
2,2023-04-01,C00001,P0160,763.50,15,2023-04,2023,4,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Repair Kits Type 1 0160,Industrial Equipment,Maintenance Kits,Repair Kits,Repair Kits Type 1,54.56,Low
3,2023-05-01,C00001,P0170,805.03,19,2023-05,2023,5,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Core Basic Type 2 0170,Industrial Equipment,Core Products,Core Basic,Core Basic Type 2,42.54,Medium
4,2023-05-01,C00001,P0160,155.10,3,2023-05,2023,5,BrightPath Distribution AG 0001,Distributor,Very Large,Logistics,Southern Europe,Central,Italy,Inbound,2023-02-01,Repair Kits Type 1 0160,Industrial Equipment,Maintenance Kits,Repair Kits,Repair Kits Type 1,54.56,Low


Final Data Quality Report

In [ ]:
def final_table_summary(df: pd.DataFrame, table_name: str) -> dict:
    return {
        "table_name": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_cell_pct": round(float(df.isna().mean().mean()), 4)}


data_quality_report = pd.DataFrame([
    final_table_summary(fact_sales_clean, "fact_sales_clean"),
    final_table_summary(dim_customer_clean, "dim_customer_clean"),
    final_table_summary(dim_product_clean, "dim_product_clean"),
    final_table_summary(fact_sf_clean, "fact_sf_clean"),
    final_table_summary(df_sales_full, "df_sales_full")])

display(data_quality_report)


,table_name,rows,columns,duplicate_rows,missing_cells,missing_cell_pct
0,fact_sales_clean,68723,8,0,0,0.00
1,dim_customer_clean,2000,10,0,0,0.00
2,dim_product_clean,180,8,0,0,0.00
3,fact_sf_clean,39508,11,0,0,0.00
4,df_sales_full,68723,24,0,0,0.00


Quick Business Sense Checks

In [21]:
monthly_sales = (
    fact_sales_clean
    .groupby("YearMonth", as_index=False)
    .agg(Sales=("Sales", "sum"), Units=("Units", "sum"), Customers=("ID_Customer", "nunique")))
monthly_sales["YearMonth"] = monthly_sales["YearMonth"].astype(str)

display(monthly_sales.head())
display(monthly_sales.tail())


,YearMonth,Sales,Units,Customers
0,2023-01,"2,775,377.56",10177,700
1,2023-02,"3,096,008.33",10804,721
2,2023-03,"4,004,158.12",14195,798
3,2023-04,"3,697,613.33",13113,781
4,2023-05,"4,033,200.89",14258,805


,YearMonth,Sales,Units,Customers
31,2025-08,"2,953,958.94",10165,612
32,2025-09,"3,918,284.05",13933,724
33,2025-10,"4,728,119.21",17215,833
34,2025-11,"5,267,593.94",19072,864
35,2025-12,"2,366,435.21",8750,555


In [ ]:
segment_summary = (
    df_sales_full
    .groupby("Customer_Segment", as_index=False)
    .agg(
        Sales=("Sales", "sum"),
        Units=("Units", "sum"),
        Customers=("ID_Customer", "nunique"),
        Products=("ID_Product", "nunique"))
    .sort_values("Sales", ascending=False))

display(segment_summary)


,Customer_Segment,Sales,Units,Customers,Products
1,Enterprise,"46,504,476.50",140393,312,180
2,Mid-Market,"39,088,542.90",149326,649,180
0,Distributor,"19,550,162.31",82091,255,180
4,SMB,"13,618,609.70",56315,602,180
3,Public Sector,"13,118,843.61",40374,175,178


In [23]:
product_group_summary = (
    df_sales_full
    .groupby("Prd_Grp_1", as_index=False)
    .agg(
        Sales=("Sales", "sum"),
        Units=("Units", "sum"),
        Customers=("ID_Customer", "nunique"),
        Products=("ID_Product", "nunique"))
    .sort_values("Sales", ascending=False))

display(product_group_summary)


,Prd_Grp_1,Sales,Units,Customers,Products
0,Automation Modules,"57,582,273.11",136488,1656,43
5,Service Contracts,"40,979,322.71",66838,1174,28
3,Premium Products,"14,545,319.10",60881,1055,24
2,Maintenance Kits,"8,311,379.14",73411,1310,30
1,Core Products,"7,553,835.39",97036,1622,39
4,Safety Solutions,"2,908,505.57",33845,909,16


Safe clean outputs

In [24]:
outputs = {
    "fact_sales_clean": fact_sales_clean,
    "dim_customer_clean": dim_customer_clean,
    "dim_product_clean": dim_product_clean,
    "fact_sf_clean": fact_sf_clean,
    "df_sales_full": df_sales_full,
    "data_quality_report": data_quality_report}

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)


Saved files:
- data_quality_report.csv
- df_sales_full.csv
- dim_customer_clean.csv
- dim_product_clean.csv
- fact_sales_clean.csv
- fact_sf_clean.csv
